# From Lab to Launch: a base model behavior
### One extraction task, two models, a harder second announcement

We start with **Qwen2.5 0.5B Base** running locally through Ollama. First, inspect a simple announcement. Then try a complex announcement and compare the outcomes.

**Our evaluation:** Does the response stay within the supplied text? Does it preserve missing information and ambiguity? Does it follow the requested format?

These are fixed, user-supplied excerpts. Models receive the pasted text, not the source pages. No browsing, retrieval, tools, or conversation history are provided.

**Default: run live.** Complete [setup.md](setup.md), restart the kernel and run all cells of this notebook. Every inference cell sends a fresh request to the local Ollama models. Saved outputs are prior observations, not a replay mode.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "scripts" / "demo.py").is_file():
    raise RuntimeError("Start Jupyter from the repository root containing scripts/ and data/.")
sys.path.insert(0, str(ROOT / "scripts"))
from demo import BASE, INSTRUCT, OPTIONS, preflight, run_case

setup = preflight()
saved_excerpts = {
    name: (ROOT / "data" / f"{name}.txt").read_text().rstrip("\n")
    for name in ("infosecon", "times_square")
}
print("LIVE mode ready: both local models and fixed excerpts are available.")
print("Ollama version:", setup["ollama_version"])
print("Each inference cell below sends a fresh request; no saved response is loaded.")

LIVE mode ready: both local models and fixed excerpts are available.
Ollama version: 0.40.5
Each inference cell below sends a fresh request; no saved response is loaded.


## 1. Prompt: shared task

Extract the event, location, date components, time, and admission price. Mark missing information and flag conflicting year references. Use eight labeled lines.

**Before the first run:** What would a useful answer contain? What would you count as a failure?

<details><summary>How the comparison works</summary>

A base model learns text continuation during pretraining. An instruction-tuned model receives additional training to respond to requests. Either model can succeed or fail on an individual example.

The base model receives the task prepended as plain text through `/api/generate` with `raw=True`. The instruction model receives the same task text in a user message through `/api/chat`, using its chat template. This compares intended usage; it does not isolate training from input formatting.

Both use temperature 0, seed 42, a 2,048-token context, and a 256-token output cap. Fixed settings reduce variation; they do not guarantee identical results across software or hardware. A response that reaches the cap needs separate inspection for truncation.

The full task below is intentionally explicit. Clearer instructions are part of product design, but they do not guarantee the model can carry them out.

</details>


In [2]:
TASK = """Using only the supplied text, extract the event, location, complete event date, event time, and admission price.
Return exactly these eight labeled lines:
Event: ...
Location: ...
Event day: ...
Event month: ...
Event year: ...
Event time: ...
Admission price: ...
Conflicting years: ...

Rules:
- Event day is the day of the month as a number; event month is the full month name; event year is four digits.
- Use the event date, not a ticket-purchase deadline. Combine date components stated in different parts of the text.
- Mark any missing field or date component as "Not stated". Never guess missing information.
- If year references conflict, keep the supported day and month, write "Conflicting" for Event year, and list the conflicting years with their source phrases under Conflicting years. Otherwise write "None stated" there.
- For Event time, copy the stated time or time range, preserving approximate wording and the time zone if supplied. Do not invent a start time, end time, or time zone. A countdown duration is not an event time.
- Ticket purchase requirements do not state an admission price. Do not infer a price or assume admission is free.
- Give only these eight lines, without explanation outside the fields."""

print(TASK)
print("\nGeneration settings:", OPTIONS)

Using only the supplied text, extract the event, location, complete event date, event time, and admission price.
Return exactly these eight labeled lines:
Event: ...
Location: ...
Event day: ...
Event month: ...
Event year: ...
Event time: ...
Admission price: ...
Conflicting years: ...

Rules:
- Event day is the day of the month as a number; event month is the full month name; event year is four digits.
- Use the event date, not a ticket-purchase deadline. Combine date components stated in different parts of the text.
- Mark any missing field or date component as "Not stated". Never guess missing information.
- If year references conflict, keep the supported day and month, write "Conflicting" for Event year, and list the conflicting years with their source phrases under Conflicting years. Otherwise write "None stated" there.
- For Event time, copy the stated time or time range, preserving approximate wording and the time zone if supplied. Do not invent a start time, end time, or time 

## 2. Clear announcement → base model

Source provenance: [Triangle InfoSeCon event section](https://raleigh.issa.org/event/triangle-infosecon/). The fixed text below preserves the supplied wording; link styling and heading markup are removed. It includes both an event date and a ticket deadline. Read it before running the model.

In [3]:
INFOSECON = """Triangle InfoSeCon | Friday, October 30, 2026

Raleigh Convention Center

Advance Ticket Purchase Required

Buy No Later Than September 30, 2026. Details

Date:
October 30

Time:
9:00 am - 5:00 pm EDT

Event Categories:
Raleigh ISSA Chapter, Triangle InfoSeCon

Event Tags:
CyberSecurity, Raleigh ISSA, Triangle InfoSeCon

"""
assert INFOSECON.rstrip("\n") == saved_excerpts["infosecon"], "Saved and pasted excerpts differ."
print(INFOSECON)

Triangle InfoSeCon | Friday, October 30, 2026

Raleigh Convention Center

Advance Ticket Purchase Required

Buy No Later Than September 30, 2026. Details

Date:
October 30

Time:
9:00 am - 5:00 pm EDT

Event Categories:
Raleigh ISSA Chapter, Triangle InfoSeCon

Event Tags:
CyberSecurity, Raleigh ISSA, Triangle InfoSeCon




In [4]:
info_base = run_case(INFOSECON, BASE, "infosecon-base", task=TASK)

Live Ollama request: qwen2.5:0.5b-base / infosecon-base at 2026-11-10T13:25:44.141617+00:00


Event: Triangle InfoSeCon
Location: Raleigh Convention Center
Event day: Friday
Event month: October
Event year: 2026
Event time: 9:00 am - 5:00 pm EDT
Admission price: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting years: None stated
Conflicting y

<Details>**Pause: what actually happened?**

- Are the event and venue correct?
- Is the event day **30**, rather than Friday or the ticket deadline?
- Does the response preserve the stated time and leave the price unknown?
- Does it finish with exactly the eight requested fields, or repeat until the output cap?

Some useful facts can appear inside an unsuccessful response. Separate extraction errors, instruction-following errors, and truncation. A single example does not establish reliability.</Details>


## 3. More complex announcement → base model

Source provenance: [Times Square announcement](https://www.timessquarenyc.org/nye/nye-landing). This is the supplied fixed excerpt, including its countdown and conflicting years. Do not refresh the page or replace the excerpt during the demo.

Before running: identify what is stated, what is missing, and what cannot be reconciled from this text alone.

In [5]:
TIMES_SQUARE = """Countdown to NYE

066
Days

02
Hours

05
Seconds to 2027.

Every year as the clock nears midnight on December 31, the eyes of the world turn once more to the dazzling lights and bustling energy of Times Square. Anticipation runs high. New Year's Eve at the symbolic center of New York City has become more than just a celebration — it's a global tradition. The world holds its breath, and cheers as the clocks strike twelve.

As the famous New Year's Eve Ball descends atop One Times Square, countless people watch New Year's Eve in Times Square, nationwide, and throughout the world and are united in bidding collective farewell to the departing year and expressing joy and hope for the year ahead. We can't wait to see you this December for Times Square New Year's Eve 2025!"""
assert TIMES_SQUARE.rstrip("\n") == saved_excerpts["times_square"], "Saved and pasted excerpts differ."
print(TIMES_SQUARE)

Countdown to NYE

066
Days

02
Hours

05
Seconds to 2027.

Every year as the clock nears midnight on December 31, the eyes of the world turn once more to the dazzling lights and bustling energy of Times Square. Anticipation runs high. New Year's Eve at the symbolic center of New York City has become more than just a celebration — it's a global tradition. The world holds its breath, and cheers as the clocks strike twelve.

As the famous New Year's Eve Ball descends atop One Times Square, countless people watch New Year's Eve in Times Square, nationwide, and throughout the world and are united in bidding collective farewell to the departing year and expressing joy and hope for the year ahead. We can't wait to see you this December for Times Square New Year's Eve 2025!


In [6]:
times_base = run_case(TIMES_SQUARE, BASE, "times-square-base", task=TASK)

Live Ollama request: qwen2.5:0.5b-base / times-square-base at 2026-11-10T13:25:45.917363+00:00


Event: New Year's Eve Ball
Location: Times Square
Event day: 12
Event month: December
Event year: 2025
Event time: 12:00:00
Admission price: Free
Conflicting years: None stated
Conflicting years: 2025
Conflicting years: 2025

Saved run: 20261110T132546276894Z-times-square-base.json


<details>**Pause: what changed with the input?**

Check day **31**, month **December**, missing admission price, and the different year references. The text gives approximate midnight timing, not an exact start/end range or a time zone.

**Important ambiguity:** a countdown to 2027 normally refers to the transition into that year; the closing sentence advertises NYE 2025. These references warrant a flag, not an automatic choice of 2027 as the event year. A countdown duration is not a reliable date without its capture timestamp.

Identify unsupported claim before showing the instruction model.</details>


## 4. The same announcement, instruction-tuned model

Model: **Qwen2.5 0.5B Instruct**.
Keep the event text, task, and generation settings unchanged. Only the model and its intended input format change. Start a fresh request; the instruction model does not see the base model's answer.

**Prediction:** Which behavior might improve? Which failure could remain?


In [7]:
times_instruct = run_case(TIMES_SQUARE, INSTRUCT, "times-square-instruct", task=TASK)

Live Ollama request: qwen2.5:0.5b-instruct / times-square-instruct at 2026-11-10T13:25:46.289638+00:00


Event: New Year's Eve Ball
Location: Times Square
Event day: December 31
Event month: December
Event year: 2027
Event time: 23:59:59
Admission price: Not stated
Conflicting years: None stated

Explanation:
- Event day: December 31
- Event month: December
- Event year: 2027
- Event time: 23:59:59
- Admission price: Not stated
- Conflicting years: None stated

Saved run: 20261110T132547376642Z-times-square-instruct.json


## 5. Compare the evidence

The compact view below shows the **first occurrence** of each requested field and flags repeated fields. It does not repair responses or assign correctness scores. Full, unchanged responses remain above.

**Compare by requirement:** factual support, missing information, ambiguity handling, and format. Fluency alone does not settle any of these.


In [8]:
from collections import Counter
from html import escape
from IPython.display import HTML, display

FIELDS = (
    "Event", "Location", "Event day", "Event month", "Event year",
    "Event time", "Admission price", "Conflicting years",
)
cases = [
    ("InfoSeCon / base", info_base),
    ("Times Square / base", times_base),
    ("Times Square / instruct", times_instruct),
]

def compact_comparison(cases):
    parsed = []
    for label, record in cases:
        first, counts = {}, Counter()
        for line in record["output"].splitlines():
            key, separator, value = line.partition(":")
            key = key.strip()
            if separator and key in FIELDS:
                counts[key] += 1
                first.setdefault(key, value.strip())
        repeats = ", ".join(f"{key} ({count} occurrences)"
                            for key, count in counts.items() if count > 1)
        parsed.append((label, first, repeats or "None detected",
                       record["response"].get("done_reason", "Not returned")))
    headings = ["Field"] + [item[0] for item in parsed]
    rows = [[field] + [item[1].get(field, "[Field absent]") for item in parsed]
            for field in FIELDS]
    rows += [["Repeated fields"] + [item[2] for item in parsed],
             ["Finish reason"] + [item[3] for item in parsed]]
    header = "<tr>" + "".join(f"<th>{escape(value)}</th>" for value in headings) + "</tr>"
    body = "".join("<tr>" + "".join(f"<td>{escape(value)}</td>" for value in row)
                   + "</tr>" for row in rows)
    return "<table><thead>" + header + "</thead><tbody>" + body + "</tbody></table>"

display(HTML(compact_comparison(cases)))


Field,InfoSeCon / base,Times Square / base,Times Square / instruct
Event,Triangle InfoSeCon,New Year's Eve Ball,New Year's Eve Ball
Location,Raleigh Convention Center,Times Square,Times Square
Event day,Friday,12,December 31
Event month,October,December,December
Event year,2026,2025,2027
Event time,9:00 am - 5:00 pm EDT,12:00:00,23:59:59
Admission price,None stated,Free,Not stated
Conflicting years,None stated,None stated,None stated
Repeated fields,Conflicting years (28 occurrences),Conflicting years (3 occurrences),None detected
Finish reason,length,stop,stop


### Ground Truth

Reference answers for the supplied fixed excerpts, rather than independently verified current event facts.

| Field | InfoSeCon | Times Square |
| --- | --- | --- |
| Event | Triangle InfoSeCon | Times Square New Year's Eve / ball drop |
| Location | Raleigh Convention Center | Times Square, New York City; ball atop One Times Square |
| Event day | 30 | 31 |
| Event month | October | December |
| Event year | 2026 | Conflicting |
| Event time | 9:00 am - 5:00 pm EDT | Near midnight / as clocks strike twelve; no exact start/end range or time zone stated |
| Admission price | Not stated | Not stated |
| Conflicting years | None stated | 2027 in countdown versus NYE 2025 in closing sentence |

September 30, 2026 is a **purchase deadline**, not a competing event year. “Advance Ticket Purchase Required” does not supply a price. Times Square's fame does not supply an admission price either: the task permits only the excerpt



### Lesson Debrief: a better response can still fail the task

 Did the base model extract the date components and preserve missing prices? Did either model repeat fields or reach the output cap? Did instruction tuning improve any field while leaving other requirements unmet? Compare against the ground truth and inspect the full responses before drawing a conclusion.

**Prompt refinement is a lesson of its own.** We made the year-handling rule more explicit after inspecting the case. That is useful product development work. To evaluate generalization, freeze the prompt and rubric, then test new announcements that did not guide those edits. Our two examples support a teaching discussion, not a benchmark ranking.


<details><summary>Reproducibility notes</summary>

Run all cells after changing `TASK` so every comparison uses the same instructions. Keep earlier run records. Existing helper code saves exact requests, excerpt hashes, settings, Ollama version, model digests, and responses under `results/`. The comparison above only summarizes saved text; inspect full outputs for extra prose, omitted fields, and truncation.

Live execution is the default. Complete [setup.md](setup.md) before running. Each model cell sends a new Ollama request; the comparison summarizes the responses from this kernel session. Keep this notebook in its project folder beside `scripts/` and `data/`.

For a fuller comparison later, run InfoSeCon with the instruction model too. Avoid further rehearsal-driven prompt changes solely to produce a preferred winner.

</details>

References: [Ollama raw generation](https://docs.ollama.com/api/generate), [chat messages](https://docs.ollama.com/api/chat), [Qwen2.5 technical report](https://arxiv.org/abs/2412.15115).
